# Filtre 3 — le sujet v2 est-il exécutable ?

**Ce notebook ne produit aucun résultat scientifique.** Il mesure deux choses, et deux
seulement : **le temps** et **la mémoire**.

La question à laquelle il répond : *quatre bras d'entraînement tiennent-ils dans les trois
semaines restantes et les 30 h de GPU hebdomadaires ?*

## Méthode

On ne lance pas un entraînement complet pour le savoir. On mesure le **débit** sur un petit
nombre de pas, puis on extrapole. Un pas de DPO coûte le même prix qu'il y en ait 20 ou
2 000 — seul leur nombre change.

C'est aussi ce qui rend ce notebook peu coûteux : quelques minutes de GPU pour décider d'un
engagement de trois semaines.

## Ce qui a déjà été mesuré, et qu'on ne remesure pas

| Constat | Source |
| :---- | :---- |
| Les backbones sont `qwen3_5`, multimodaux, chargés en texte seul | smoke test v1 |
| Le T4 n'a pas de bf16 — `resolve_precision` retombe en fp16 | smoke test v1 |
| Le vocabulaire de 248 044 rend les logits DPO dominants en mémoire | OOM v1 |
| PEFT n'a pas de `target_modules` pour `qwen3_5` | crash v1 |
| Aya haoussa = 3 512, Uhura 791, UbuntuGuard 128 | notebook 03 |

---

### Réglages Kaggle

- **Accelerator : T4 x2** — pas P100. Pascal n'a pas de tensor cores fp16 exploitables et
  exécute nettement plus lentement le chemin 4-bit du projet.
- **Internet activé** — les entrées modèle Kaggle sont des pointeurs vers le Hub.
- **Secrets** : `HF_TOKEN`, `GITHUB_PAT`.

## 0. Mise en place

In [ ]:
!pip install -q -U "transformers>=5.12.1" trl peft bitsandbytes accelerate datasets

In [ ]:
import os, sys
from pathlib import Path

# Le code arrive par Kaggle Datasets, pas par un clone git. Raison mesuree sur deux runs
# reels: un secret Kaggle ne peut pas etre attache par l'API, et un push par API efface les
# attachements faits dans l'editeur. Un notebook dependant d'un secret n'est donc pas
# soumettable en fire-and-forget.
#
# Deux datasets separes plutot qu'un: le code fait 0,1 Mo et se republie a chaque
# iteration, les donnees font 31 Mo et n'ont pas bouge depuis qu'on les a versionnees.
# Les reunir faisait reteleverser 31 Mo pour rien a chaque cycle.

ENTREE = Path("/kaggle/input")

def localiser(marqueur, defaut_local):
    """Trouve le dataset qui contient `marqueur`, ou retombe sur le depot local.

    On remonte au premier segment sous /kaggle/input plutot que de compter des niveaux de
    parents: l'arithmetique d'indices se trompait d'un cran et renvoyait /kaggle/input
    lui-meme, ce qui aurait casse l'import sans dire pourquoi.
    """
    if ENTREE.exists():
        for trouve in ENTREE.glob(marqueur):
            return ENTREE / trouve.relative_to(ENTREE).parts[0]
    return defaut_local

LOCAL = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "data.py").exists())
ROOT = localiser("*/src/data.py", LOCAL)
DATA = localiser("*/data/Ubuntu_guard_test_crosslingual.jsonl", LOCAL)

sys.path.insert(0, str(ROOT))
print("code    :", ROOT)
print("donnees :", DATA)

# /kaggle/input est en lecture seule: tout ce qu'on ecrit va dans /kaggle/working.
SORTIE = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("results")
SORTIE.mkdir(parents=True, exist_ok=True)
print("sorties :", SORTIE)

# HF_TOKEN reste lu s'il est la, mais n'est pas bloquant: tous les jeux sont publics.
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN: present")
except Exception:
    print("HF_TOKEN: absent -- jeux publics, on continue")

### Contrôle de version

Si le commit affiché n'est pas le dernier, tout ce qui suit mesure du code périmé.
Sur Kaggle, le système de fichiers survit à un redémarrage de kernel : sans `git pull`
explicite, un clone existant ne bougerait pas.

In [ ]:
import time, json
import torch

print("GPU        :", torch.cuda.get_device_name(0))
print("VRAM totale:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2), "Go")
print("bf16       :", torch.cuda.is_bf16_supported())
print("GPUs       :", torch.cuda.device_count())

import transformers, trl, peft
print(f"\ntransformers {transformers.__version__} | trl {trl.__version__} | peft {peft.__version__}")
assert transformers.__version__ >= "5.12", "qwen3_5 exige transformers >= 5.12.1"

## 1. Données — les mêmes que le notebook 03

Aucune resplit ici : les fonctions sont celles déjà testées, avec la même graine.

In [ ]:
import yaml
from datasets import load_dataset

from src.data import (
    build_aya_sft_examples, build_preference_pairs, build_uhura_pairs,
    load_ubuntuguard_rows, split_by_base_stem,
)

config = yaml.safe_load(open(ROOT / "config.yaml"))
LANGUE = config["sft"]["language"]

aya = load_dataset("CohereLabs/aya_dataset", split="train")
aya_ha = aya.filter(lambda r: r["language"] == LANGUE)      # filtrer avant list()
sft_examples = build_aya_sft_examples(list(aya_ha), LANGUE)

uhura = load_dataset("masakhane/uhura-truthfulqa", "ha_generation", split="test")
ug = load_ubuntuguard_rows(DATA / "data" / "Ubuntu_guard_test_crosslingual.jsonl")
dpo_pairs = build_uhura_pairs(list(uhura), LANGUE) + [
    p for p in build_preference_pairs(ug) if p["language"] == LANGUE
]

tr_sft, ev_sft = split_by_base_stem(sft_examples)
tr_dpo, ev_dpo = split_by_base_stem(dpo_pairs)

print(f"SFT : {len(tr_sft)} train / {len(ev_sft)} eval")
print(f"DPO : {len(tr_dpo)} train / {len(ev_dpo)} eval")
assert not ({p["base_stem"] for p in tr_sft} & {p["base_stem"] for p in dpo_pairs}), \
    "SFT et DPO partagent une question"
print("\ncontamination SFT <-> DPO : 0")

## 2. Débit du SFT

`max_steps` volontairement petit. On veut des secondes par pas, pas un modèle entraîné.

La mémoire est remise à zéro avant chaque mesure pour que `max_memory_allocated` reflète
bien cette phase et non ce qui précède.

In [ ]:
from src.train import run_sft

PAS_MESURE = 8

torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
t0 = time.time()
trainer_sft = run_sft(config, tr_sft, max_steps=PAS_MESURE)
duree_sft = time.time() - t0
vram_sft = torch.cuda.max_memory_allocated() / 1e9

par_pas_sft = duree_sft / PAS_MESURE
print(f"\nSFT  : {duree_sft:.0f}s pour {PAS_MESURE} pas -> {par_pas_sft:.1f}s/pas")
print(f"       VRAM crete : {vram_sft:.2f} Go")

In [ ]:
# Liberer avant la phase DPO : sans cela la mesure suivante inclut ce modele.
del trainer_sft
import gc; gc.collect(); torch.cuda.empty_cache()
print("VRAM apres liberation :", round(torch.cuda.memory_allocated() / 1e9, 2), "Go")

## 3. Débit du DPO

Le DPO est structurellement plus lourd : il tient un modèle de référence figé en plus du
modèle entraîné, et calcule des logits pour `chosen` **et** `rejected`. Avec un vocabulaire
de 248 044, ce sont ces logits qui dominent la mémoire, pas les poids.

In [ ]:
from src.train import run_dpo

torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
t0 = time.time()
trainer_dpo = run_dpo(config, tr_dpo, max_steps=PAS_MESURE)
duree_dpo = time.time() - t0
vram_dpo = torch.cuda.max_memory_allocated() / 1e9

par_pas_dpo = duree_dpo / PAS_MESURE
print(f"\nDPO  : {duree_dpo:.0f}s pour {PAS_MESURE} pas -> {par_pas_dpo:.1f}s/pas")
print(f"       VRAM crete : {vram_dpo:.2f} Go")

## 4. Extrapolation

Un pas d'optimisation traite `batch_size x gradient_accumulation_steps` exemples. Le nombre
de pas d'une vraie run se déduit donc du volume, du batch effectif et du nombre d'époques.

In [ ]:
import math

def pas_reels(n_exemples, cfg, epochs):
    lot_effectif = cfg["batch_size"] * cfg["gradient_accumulation_steps"]
    return math.ceil(n_exemples / lot_effectif) * epochs

n_sft = pas_reels(len(tr_sft), config["sft"], config["sft"]["num_epochs"])
n_dpo = pas_reels(len(tr_dpo), config["dpo"], config["dpo"]["num_epochs"])

h_sft = n_sft * par_pas_sft / 3600
h_dpo = n_dpo * par_pas_dpo / 3600
h_bras = h_sft + h_dpo

print(f"SFT  : {n_sft:>4} pas -> {h_sft:.2f} h")
print(f"DPO  : {n_dpo:>4} pas -> {h_dpo:.2f} h")
print(f"\nUN BRAS (SFT + DPO)   : {h_bras:.2f} h")
print(f"DEUX bras alignes     : {2 * h_bras:.2f} h   <- A2 et A3, le claim")
print(f"+ chargement modeles  : ~{4 * 50 / 60:.0f} min de telechargement par session froide")

In [ ]:
BUDGET_HEBDO = 30
SEMAINES = 3
GRAINES = 3

total_1_graine = 2 * h_bras
total_n_graines = GRAINES * total_1_graine

print(f"budget disponible      : {BUDGET_HEBDO * SEMAINES} h ({BUDGET_HEBDO} h x {SEMAINES} semaines)")
print(f"deux bras, 1 graine    : {total_1_graine:.1f} h")
print(f"deux bras, {GRAINES} graines   : {total_n_graines:.1f} h")
print()
marge = BUDGET_HEBDO * SEMAINES - total_n_graines
print(f"marge restante         : {marge:.1f} h")
print()
if total_n_graines < BUDGET_HEBDO:
    print("VERDICT : confortable. Les 3 graines tiennent dans UNE semaine de quota.")
elif total_n_graines < BUDGET_HEBDO * SEMAINES * 0.5:
    print("VERDICT : jouable. Moins de la moitie du budget, marge pour les reprises.")
elif total_n_graines < BUDGET_HEBDO * SEMAINES:
    print("VERDICT : serre. Prevoir 1 graine d'abord, les autres si le temps le permet.")
else:
    print("VERDICT : hors budget. Reduire: moins d'epoques, moins d'exemples, ou 1 seule graine.")

## 5. Trace pour l'estimation compute

Livrable contractuel reporté depuis la semaine 5. Ces chiffres sont mesurés, pas estimés.

In [ ]:
mesures = {
    "gpu": torch.cuda.get_device_name(0),
    "vram_totale_go": round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2),
    "bf16_supporte": torch.cuda.is_bf16_supported(),
    "sft": {
        "exemples": len(tr_sft),
        "s_par_pas": round(par_pas_sft, 2),
        "vram_crete_go": round(vram_sft, 2),
        "pas_run_complete": n_sft,
        "heures": round(h_sft, 2),
    },
    "dpo": {
        "paires": len(tr_dpo),
        "s_par_pas": round(par_pas_dpo, 2),
        "vram_crete_go": round(vram_dpo, 2),
        "pas_run_complete": n_dpo,
        "heures": round(h_dpo, 2),
    },
    "un_bras_heures": round(h_bras, 2),
    "deux_bras_trois_graines_heures": round(total_n_graines, 2),
}
pass
Path(SORTIE / "compute_estimate.json").write_text(json.dumps(mesures, indent=2), encoding="utf-8")
print(json.dumps(mesures, indent=2))

---

## Ce que ce notebook décide

| Observation | Suite |
| :---- | :---- |
| Deux bras × 3 graines < 30 h | plan inchangé, on lance les quatre bras |
| Entre 30 et 90 h | une graine d'abord, les autres si le temps reste |
| Au-delà de 90 h | réduire : époques, volume, ou renoncer aux graines multiples |
| OOM sur le SFT | réduire `max_seq_length`, le SFT n'a pas besoin de 2 560 |
| OOM sur le DPO | déjà à `batch_size: 1` — reste à baisser la longueur de séquence |

**Point d'attention sur les graines.** Trois graines ne sont pas un luxe : le claim est une
*différence* entre deux bras. Une différence mesurée sur une seule graine ne se distingue
pas du bruit d'initialisation. Si le budget ne permet qu'une graine, il faudra le déclarer
comme limitation, pas le passer sous silence.